In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.sparse import diags, identity
from scipy.sparse.linalg import splu
    
def bs_fd(S0=100.0, K=100.0, r=0.03, q=0.0, sigma=0.3, T=1.0,
          J=100, N=50, theta=0.5, c=5.0, Smax=None):
    """
    J     : number of space INTERVALS, with J+1 gridpoints
    N     : number of time steps
    theta : 0 explicit, 1 implicit, 0.5 Crank-Nicolson
    c     : numbr of standard deviations of log S_T to compute Smax
    Smax  : override the automatic domain if you want a fixed one

    Returns (price_at_S0, S_grid, V_grid, stable_flag)
    """

    # Compute Smax in case no Smax is specified
    if Smax is None:
        Smax = max(S0, K) * np.exp((r - q - sigma**2/2)*T + c*sigma*np.sqrt(T))
    Smin = 0.0

    # Forcing S0 to fall in a node exactly. 
    j0 = max(1, int(round(S0 / (Smax/J))))   # node index of S0
    dS = S0 / j0                             # time step size
    Smax = J * dS                            # adjusted domain

    S = np.linspace(Smin, Smax, J + 1)   #creating the S+1 gridpoints  
    dt = T/N    #step size

    #stability check for the explicit method
    stable = dt <= dS**2/(sigma**2 * Smax**2) 

    m = J - 1
    Si = S[1:-1]

    #call payoff
    u = np.maximum(Si - K, 0.0)

    # coefficient vectors
    f = -0.5*sigma**2 * Si**2 / dS**2
    g = -(r - q) * Si / (2*dS)
    h = r * np.ones(m)

    F = f + g          
    G = 2*f - h
    H = f - g  

    # M = -L  :  sub = H, diag = -G, super = F
    M = diags([H[1:], -G, F[:-1]], offsets=[-1, 0, 1], format="csc")
    I = identity(m, format="csc")

    lhsmat = I/dt + theta*M
    rhsmat = I/dt - (1 - theta)*M

    # factor once outside the time loop (lhsmat never changes)
    if theta != 0.0:
        lu = splu(lhsmat.tocsc())

    # ---- time stepping ----
    for n in range(1, N + 1):
        rhsvec = rhsmat @ u

        # boundary values at the new (tau_n) and old (tau_{n-1}) time levels
        umin_now = 0.0
        umin_pre = 0.0
        umax_now = Smax*np.exp(-q*n*dt)     - K*np.exp(-r*n*dt)
        umax_pre = Smax*np.exp(-q*(n-1)*dt) - K*np.exp(-r*(n-1)*dt)

        rhsvec[0]  -= theta*H[0]*umin_now  + (1 - theta)*H[0]*umin_pre
        rhsvec[-1] -= theta*F[-1]*umax_now + (1 - theta)*F[-1]*umax_pre

        if theta == 0.0:
            u = dt*rhsvec           # lhsmat is I/dt, so no solve needed
        else:
            u = lu.solve(rhsvec)

    # ---- reassemble with boundaries and read off the price ----
    V = np.concatenate([[0.0], u, [Smax*np.exp(-q*T) - K*np.exp(-r*T)]])
    price = np.interp(S0, S, V)
    return price, S, V, stable


# ----------------------------------------------------------------------
# driver: reproduce the convergence table
# ----------------------------------------------------------------------
if __name__ == "__main__":
    S0, K, r, q, sigma, T = 100.0, 100.0, 0.03, 0.0, 0.3, 1.0
    exact = bs_price(S0, K, r, T, sigma, q, "call")
    print(f"Exact European call: {exact:.9f}\n")

    kmax = 5
    J_list = 50 * 2**np.arange(kmax + 1)
    N_list = 25 * 2**np.arange(kmax + 1)

    for theta, name in [(0.0, "explicit"), (1.0, "implicit"), (0.5, "Crank-Nicolson")]:
        print(f"--- theta = {theta}  ({name}) ---")
        print(f"{'J':>6} {'N':>6} {'price':>14} {'error':>12} {'ratio':>8}")
        prev = None
        for J, N in zip(J_list, N_list):
            p, _, _, stable = bs_fd(S0, K, r, q, sigma, T, J=J, N=N, theta=theta)
            err = exact - p
            ratio = prev/err if prev is not None else np.nan
            flag = "" if (theta > 0 or stable) else "  <- UNSTABLE"
            print(f"{J:6d} {N:6d} {p:14.6f} {err:12.3e} {ratio:8.3f}{flag}")
            prev = err
        print()